# 01 - Carga y Exploración Inicial - OULAD

**Objetivo específico 1:** Identificar las variables académicas, conductuales y temporales
disponibles en el dataset, mediante técnicas de preprocesamiento y análisis exploratorio,
con el fin de establecer su pertinencia para el estudio del abandono en un entorno LMS.

Proyecto: Modelo predictivo del abandono estudiantil en entornos LMS (OULAD)

**Estado:** Validado y ejecutado. Ver sección 7 (Hallazgos y observaciones) para el
resumen de decisiones metodológicas derivadas de esta exploración.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga de las 7 tablas

In [ ]:
tablas = {
    'courses': RAW_DIR / 'courses.csv',
    'assessments': RAW_DIR / 'assessments.csv',
    'vle': RAW_DIR / 'vle.csv',
    'studentInfo': RAW_DIR / 'studentInfo.csv',
    'studentRegistration': RAW_DIR / 'studentRegistration.csv',
    'studentAssessment': RAW_DIR / 'studentAssessment.csv',
    'studentVle': RAW_DIR / 'studentVle.csv',  # tabla mas pesada (~10M filas)
}

dfs = {}
for nombre, ruta in tablas.items():
    dfs[nombre] = pd.read_csv(ruta)
    print(f"{nombre:22s} -> shape: {dfs[nombre].shape}")

## 2. Vista general por tabla (dtypes + nulos)

In [ ]:
def resumen_tabla(df, nombre):
    print(f"\n{'='*60}\n{nombre}\n{'='*60}")
    resumen = pd.DataFrame({
        'dtype': df.dtypes,
        'n_nulos': df.isnull().sum(),
        '%_nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'n_unicos': df.nunique()
    })
    print(resumen)
    return resumen

for nombre, df in dfs.items():
    resumen_tabla(df, nombre)

**Nota:** `assessments` presenta 5.34% de nulos en `date`. Corresponde a evaluaciones
de tipo *Exam* (examen final), cuya fecha no siempre esta fijada de antemano en el
dataset original. No se elimina esta informacion en esta fase; se documenta para
tratarla explicitamente en `02_feature_engineering.ipynb` al construir variables de
desempeno academico inicial (actividad A2.5).

## 3. Variable objetivo: `studentRegistration`

El abandono se define (segun anteproyecto, seccion 8.1.3, A3.1) como la presencia de un
valor no nulo en `date_unregistration`.

In [ ]:
reg = dfs['studentRegistration']
reg.head()

In [ ]:
reg['abandono'] = reg['date_unregistration'].notnull().astype(int)

n_total = len(reg)
n_abandono = reg['abandono'].sum()
pct_abandono = n_abandono / n_total * 100

print(f"Total registros (estudiante x modulo x presentacion): {n_total}")
print(f"Registros con abandono (retiro voluntario): {n_abandono} ({pct_abandono:.2f}%)")
print(f"Registros sin abandono: {n_total - n_abandono} ({100 - pct_abandono:.2f}%)")

In [ ]:
reg['abandono'].value_counts().plot(kind='bar', title='Distribucion de la variable objetivo (abandono)')

**Resultado obtenido:** 30.90% de abandono (10,072 de 32,593 registros) frente a 69.10%
de no abandono.

**Observacion metodologica:** este es un **desbalance de clases moderado**, no extremo
(no es un escenario 95/5). Sin embargo, si justifica aplicar la actividad **A3.4** de la
metodologia (pesos por clase, remuestreo o ajuste de umbral), y evaluar el modelo con
metricas sensibles al desbalance (sensibilidad, F1-score), tal como se planteo en el
anteproyecto - no solo con exactitud.

## 4. Verificacion de llaves de union entre tablas

Las llaves principales para relacionar las tablas son:
- `code_module` + `code_presentation` (identifica un curso-presentacion)
- `id_student` (identifica al estudiante)


In [ ]:
# Verificar que id_student en studentInfo coincide con studentRegistration
ids_info = set(dfs['studentInfo']['id_student'].unique())
ids_reg = set(dfs['studentRegistration']['id_student'].unique())

print(f"Estudiantes unicos en studentInfo: {len(ids_info)}")
print(f"Estudiantes unicos en studentRegistration: {len(ids_reg)}")
print(f"Diferencia (en info pero no en reg): {len(ids_info - ids_reg)}")
print(f"Diferencia (en reg pero no en info): {len(ids_reg - ids_info)}")

In [ ]:
# Verificar llave compuesta code_module + code_presentation en courses vs studentInfo
cursos = set(dfs['courses'].apply(lambda r: (r['code_module'], r['code_presentation']), axis=1))
cursos_info = set(dfs['studentInfo'].apply(lambda r: (r['code_module'], r['code_presentation']), axis=1))

print(f"Modulo-presentaciones en courses: {len(cursos)}")
print(f"Modulo-presentaciones en studentInfo: {len(cursos_info)}")
print(f"Coinciden: {cursos == cursos_info}")

**Resultado obtenido:** 0 diferencias en ambos sentidos entre `studentInfo` y
`studentRegistration` (28,785 estudiantes unicos en ambas). Las 22 combinaciones
`code_module` + `code_presentation` coinciden exactamente entre `courses` y `studentInfo`.

**Conclusion:** las llaves de union son consistentes. Los joins del Objetivo 2 pueden
construirse sin riesgo de perdida de filas por descuadre de llaves.

**Observacion importante para la documentacion del dataset analitico (Resultado
esperado #1 del anteproyecto):** `studentInfo` tiene **32,593 filas** pero solo
**28,785 estudiantes unicos**. Esto significa que la unidad de analisis real es
**estudiante x modulo x presentacion**, no el estudiante como individuo - hay
estudiantes que aparecen en mas de un modulo o repitieron una presentacion. Esta
distincion debe quedar explicita en la documentacion del dataset final, ya que afecta
como se interpretan las variables derivadas (p. ej., "regularidad de participacion" se
calcula por inscripcion, no por persona).

## 5. Vista preliminar de `studentVle` (interaccion)

Esta es la tabla mas pesada y la base para las variables conductuales/temporales
del Objetivo 2.

In [ ]:
svle = dfs['studentVle']
print(f"Shape: {svle.shape}")
print(f"Rango de 'date' (dias relativos al inicio del curso): {svle['date'].min()} a {svle['date'].max()}")
svle.head()

In [ ]:
# Proporcion de interacciones registradas antes del inicio formal del curso (date < 0)
n_pre_inicio = (svle['date'] < 0).sum()
pct_pre_inicio = n_pre_inicio / len(svle) * 100
print(f"Interacciones con date < 0 (antes del inicio formal del curso): {n_pre_inicio} ({pct_pre_inicio:.2f}%)")

**Resultado obtenido:** el rango de `date` va de **-25 a 269** dias. Los valores
negativos corresponden a interacciones registradas *antes* del inicio oficial del curso
(por ejemplo, revisar el syllabus o materiales introductorios ya publicados).

**Observacion metodologica para el Objetivo 2 (A2.2 - definicion de ventanas
temporales):** al definir las ventanas tempranas (p. ej., primeras 4-6 semanas), debe
decidirse explicitamente si:
1. Se excluyen los dias `date < 0` del calculo de variables de interaccion temprana, o
2. Se incluyen como una senal adicional (p. ej., "interaccion anticipada" como posible
   indicador de compromiso).

Esta decision debe documentarse y justificarse en el notebook de feature engineering,
ya que afecta directamente la definicion de "primera semana" de actividad de cada
estudiante.

## 6. Verificacion de llaves entre `studentVle` y `studentInfo`

Antes de pasar al feature engineering, se confirma que los estudiantes con registros de
interaccion estan contenidos en la poblacion de `studentInfo`.

In [ ]:
ids_svle = set(svle['id_student'].unique())

print(f"Estudiantes unicos en studentVle: {len(ids_svle)}")
print(f"Estudiantes en studentVle pero no en studentInfo: {len(ids_svle - ids_info)}")
print(f"Estudiantes en studentInfo sin ningun registro en studentVle: {len(ids_info - ids_svle)}")

**Nota:** si el segundo conteo (estudiantes en `studentInfo` sin registros en
`studentVle`) es mayor que cero, esos estudiantes representan candidatos naturales al
**abandono implicito por inactividad total**, mencionado en el anteproyecto (seccion 8,
introduccion a la Metodologia) como indicador complementario al retiro formal. Se
retoma esta poblacion en el Objetivo 2 / actividad A3.1.

## 7. Hallazgos y observaciones - resumen para documentacion

Consolidado de las decisiones metodologicas que este notebook deja establecidas antes
de avanzar al feature engineering (Objetivo 2):

1. **Variable objetivo:** 30.90% de abandono (10,072 / 32,593). Desbalance moderado;
   se aplicaran estrategias de mitigacion en A3.4 y metricas sensibles al desbalance
   (sensibilidad, F1-score) en A3.6.
2. **Integridad de llaves:** sin diferencias entre `studentInfo` y
   `studentRegistration` (28,785 estudiantes); 22 modulo-presentaciones coinciden
   exactamente entre `courses` y `studentInfo`.
3. **Unidad de analisis:** el dataset opera a nivel de **estudiante x modulo x
   presentacion** (32,593 filas), no de estudiante unico (28,785 personas). Debe
   documentarse asi en el dataset analitico final.
4. **`assessments.date`:** 5.34% de nulos, correspondientes a evaluaciones tipo
   *Exam*. Se trata explicitamente en A2.5.
5. **`studentVle.date`:** rango de -25 a 269 dias. Existen interacciones previas al
   inicio formal del curso; se decide el tratamiento de `date < 0` en la definicion de
   ventanas tempranas (A2.2).
6. **Abandono implicito:** se verifico la existencia (o ausencia) de estudiantes en
   `studentInfo` sin ningun registro en `studentVle`, como insumo para la exploracion
   de inactividad prolongada mencionada en el anteproyecto.

## 8. Proximos pasos

- Pasar al notebook `02_feature_engineering.ipynb` (Objetivo 2): construccion de
  variables de interaccion, continuidad, inactividad y desempeno academico inicial,
  incorporando las decisiones documentadas en la seccion 7.
